In [0]:
import os
os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/workspace/default/flight_data/mllib_tmp"

import gc
import mlflow
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

FLIGHT_FEATURES = "gold_flight_features"
WEATHER_FEATURES = "gold_weather_features"
JOINED_TABLE = "gold_joined_features"
MODEL_VOLUME_PATH = "/Volumes/workspace/default/flight_data/models"

SEED = 42
print("Setup done.")

Setup done.


In [0]:
flights = spark.table(FLIGHT_FEATURES)
weather = spark.table(WEATHER_FEATURES)

print(f"Flight rows : {flights.count():,}")
print(f"Weather rows: {weather.count():,}")

joined = (
    flights.alias("f")
    .join(
        weather.alias("w"),
        (F.col("f.Origin") == F.col("w.airport")) &
        (F.col("f.flight_hour_ts") == F.col("w.obs_hour_ts")),
        how="left"
    )
    .select(
        "f.label",
        "f.Reporting_Airline", "f.Origin", "f.DepTimeCategory",
        "f.Month", "f.DayOfWeek", "f.DepHour",
        "f.IsWeekend", "f.IsFixedHoliday", "f.IsHolidayWindow",
        "w.temp", "w.prcp", "w.wspd",
        "w.is_precipitating", "w.heavy_precip",
        "w.freezing", "w.high_wind", "w.very_high_wind",
        "w.severe_weather",
    )
)

joined_count = joined.count()
matched_count = joined.filter(F.col("temp").isNotNull()).count()

print(f"\nJoined rows: {joined_count:,}")
print(f"Rows with weather match: {matched_count:,}")
print(f"Match rate: {100.0 * matched_count / joined_count:.2f}%")

Flight rows : 302,192
Weather rows: 122,640

Joined rows: 302,192
Rows with weather match: 301,653
Match rate: 99.82%


In [0]:
joined_clean = joined.na.drop(subset=["temp", "prcp", "wspd"])

(
    joined_clean.write
                .format("delta")
                .mode("overwrite")
                .option("overwriteSchema", "true")
                .saveAsTable(JOINED_TABLE)
)

print(f"Saved {JOINED_TABLE} with {spark.table(JOINED_TABLE).count():,} rows.")

Saved gold_joined_features with 301,653 rows.


In [0]:
insight_a = spark.sql(f"""
    SELECT
        Origin,
        CASE WHEN is_precipitating = 1 THEN 'Precipitating' ELSE 'Clear' END AS weather_state,
        COUNT(*)                                                 AS n_flights,
        ROUND(100.0 * SUM(label) / COUNT(*), 2)                  AS delay_rate_pct
    FROM {JOINED_TABLE}
    WHERE Origin IN ('JFK', 'LGA', 'EWR', 'BUF', 'ALB')
    GROUP BY Origin, is_precipitating
    ORDER BY Origin, weather_state
""")
insight_a.show(truncate=False)

(
    insight_a.write
             .format("delta")
             .mode("overwrite")
             .option("overwriteSchema", "true")
             .saveAsTable("insight_precip_vs_clear")
)

+------+-------------+---------+--------------+
|Origin|weather_state|n_flights|delay_rate_pct|
+------+-------------+---------+--------------+
|ALB   |Clear        |11268    |21.43         |
|ALB   |Precipitating|973      |25.08         |
|BUF   |Clear        |18782    |20.33         |
|BUF   |Precipitating|1411     |30.83         |
|JFK   |Clear        |96075    |23.21         |
|JFK   |Precipitating|6692     |37.45         |
|LGA   |Clear        |120195   |24.82         |
|LGA   |Precipitating|7526     |36.54         |
+------+-------------+---------+--------------+



In [0]:
insight_b = spark.sql(f"""
    SELECT
        CASE
            WHEN wspd < 10  THEN '1_calm (<10 km/h)'
            WHEN wspd < 20  THEN '2_light (10-20)'
            WHEN wspd < 30  THEN '3_moderate (20-30)'
            WHEN wspd < 50  THEN '4_strong (30-50)'
            ELSE                 '5_very_strong (50+)'
        END                                                      AS wind_bucket,
        COUNT(*)                                                 AS n_flights,
        ROUND(100.0 * SUM(label) / COUNT(*), 2)                  AS delay_rate_pct
    FROM {JOINED_TABLE}
    GROUP BY wind_bucket
    ORDER BY wind_bucket
""")
insight_b.show(truncate=False)

(
    insight_b.write
             .format("delta")
             .mode("overwrite")
             .option("overwriteSchema", "true")
             .saveAsTable("insight_wind_buckets")
)

+-------------------+---------+--------------+
|wind_bucket        |n_flights|delay_rate_pct|
+-------------------+---------+--------------+
|1_calm (<10 km/h)  |74220    |19.12         |
|2_light (10-20)    |118116   |23.80         |
|3_moderate (20-30) |77430    |25.77         |
|4_strong (30-50)   |31347    |33.89         |
|5_very_strong (50+)|540      |41.85         |
+-------------------+---------+--------------+



In [0]:
insight_c = spark.sql(f"""
    SELECT
        Reporting_Airline                                        AS carrier,
        ROUND(100.0 * AVG(CASE WHEN severe_weather = 0 THEN label END), 2) AS delay_rate_normal_pct,
        ROUND(100.0 * AVG(CASE WHEN severe_weather = 1 THEN label END), 2) AS delay_rate_severe_pct,
        COUNT(CASE WHEN severe_weather = 1 THEN 1 END)           AS severe_weather_flights
    FROM {JOINED_TABLE}
    GROUP BY Reporting_Airline
    HAVING COUNT(CASE WHEN severe_weather = 1 THEN 1 END) > 100
    ORDER BY (delay_rate_severe_pct - delay_rate_normal_pct) DESC
""")
insight_c.show(20, truncate=False)

(
    insight_c.write
             .format("delta")
             .mode("overwrite")
             .option("overwriteSchema", "true")
             .saveAsTable("insight_carrier_severe_weather")
)

+-------+---------------------+---------------------+----------------------+
|carrier|delay_rate_normal_pct|delay_rate_severe_pct|severe_weather_flights|
+-------+---------------------+---------------------+----------------------+
|UA     |24.37                |50.9                 |167                   |
|OO     |22.16                |46.34                |123                   |
|YX     |22.81                |46.62                |695                   |
|WN     |21.5                 |44.44                |396                   |
|B6     |26.61                |47.88                |614                   |
|DL     |22.96                |43.58                |771                   |
|AA     |23.69                |39.9                 |401                   |
+-------+---------------------+---------------------+----------------------+



In [0]:
combined = spark.table(JOINED_TABLE).select(
    "label",
    "Reporting_Airline", "Origin", "DepTimeCategory",
    "Month", "DayOfWeek", "DepHour",
    "IsWeekend", "IsFixedHoliday", "IsHolidayWindow",
    "temp", "prcp", "wspd",
    "is_precipitating", "heavy_precip", "freezing",
    "high_wind", "very_high_wind", "severe_weather",
)

print(f"Combined dataset rows: {combined.count():,}")

train_df, test_df = combined.randomSplit([0.8, 0.2], seed=SEED)
print(f"Train: {train_df.count():,} | Test: {test_df.count():,}")

Combined dataset rows: 301,653
Train: 241,480 | Test: 60,173


In [0]:
categorical_cols = ["Reporting_Airline", "Origin", "DepTimeCategory"]
numeric_cols = [
    "Month", "DayOfWeek", "DepHour",
    "IsWeekend", "IsFixedHoliday", "IsHolidayWindow",
    "temp", "prcp", "wspd",
    "is_precipitating", "heavy_precip", "freezing",
    "high_wind", "very_high_wind", "severe_weather",
]

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_cols
]

encoder = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in categorical_cols],
    outputCols=[f"{c}_ohe" for c in categorical_cols],
    handleInvalid="keep"
)

assembler = VectorAssembler(
    inputCols=[f"{c}_ohe" for c in categorical_cols] + numeric_cols,
    outputCol="features",
    handleInvalid="keep"
)

preprocessing_stages = indexers + [encoder, assembler]
print(f"Preprocessing stages: {len(preprocessing_stages)}")

Preprocessing stages: 5


In [0]:
auc_eval = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
accuracy_eval = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")
f1_eval = MulticlassClassificationEvaluator(labelCol="label", metricName="f1")
precision_eval = MulticlassClassificationEvaluator(labelCol="label", metricName="weightedPrecision")
recall_eval = MulticlassClassificationEvaluator(labelCol="label", metricName="weightedRecall")

def evaluate_model(model, test_data, model_name):
    preds = model.transform(test_data)
    return {
        "model":     model_name,
        "roc_auc":   auc_eval.evaluate(preds),
        "accuracy":  accuracy_eval.evaluate(preds),
        "precision": precision_eval.evaluate(preds),
        "recall":    recall_eval.evaluate(preds),
        "f1":        f1_eval.evaluate(preds),
    }

In [0]:
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=50,
    maxDepth=8,
    seed=SEED
)

rf_combined_pipeline = Pipeline(stages=preprocessing_stages + [rf])

with mlflow.start_run(run_name="RandomForest_Combined"):
    rf_combined = rf_combined_pipeline.fit(train_df)
    rf_combined_metrics = evaluate_model(rf_combined, test_df, "RandomForest_Combined")
    for k, v in rf_combined_metrics.items():
        if k != "model":
            mlflow.log_metric(k, v)
    mlflow.log_param("features_source", "flights + weather")

print(rf_combined_metrics)
rf_combined.write().overwrite().save(f"{MODEL_VOLUME_PATH}/random_forest_combined_pipeline")

del rf_combined_pipeline
gc.collect()
print("Combined model saved.")

{'model': 'RandomForest_Combined', 'roc_auc': 0.6808462771995389, 'accuracy': 0.7635816728433018, 'precision': 0.7622001554554416, 'recall': 0.7635816728433018, 'f1': 0.67067397590925}
Combined model saved.


In [0]:
rf_flights_only_roc_auc = 0.6599212116709646

lift = rf_combined_metrics["roc_auc"] - rf_flights_only_roc_auc

print("Random Forest Comparison")
print("=" * 50)
print(f"Flights-only ROC-AUC (Notebook 04) : {rf_flights_only_roc_auc:.4f}")
print(f"Flights+Weather ROC-AUC (this NB)  : {rf_combined_metrics['roc_auc']:.4f}")
print(f"Lift from adding weather           : {lift:+.4f}")
print("=" * 50)
print(f"Accuracy  : {rf_combined_metrics['accuracy']:.4f}")
print(f"F1        : {rf_combined_metrics['f1']:.4f}")
print(f"Precision : {rf_combined_metrics['precision']:.4f}")
print(f"Recall    : {rf_combined_metrics['recall']:.4f}")

Random Forest Comparison
Flights-only ROC-AUC (Notebook 04) : 0.6599
Flights+Weather ROC-AUC (this NB)  : 0.6808
Lift from adding weather           : +0.0209
Accuracy  : 0.7636
F1        : 0.6707
Precision : 0.7622
Recall    : 0.7636
